# Topic 3: RDD Actions

> **Phase 2 → Week 2 → Topic 3**

---

## The ATM Analogy

You set up all your bank transactions during the day (transformations — lazy):
- Transfer 500 to rent
- Pay electricity bill
- Buy groceries

Nothing actually happens until you hit **Confirm** (Action). Then the bank
executes all steps and gives you back a **result** (your balance, receipt).

Spark Actions are the **Confirm button** — they trigger execution and return a result.

---

## Key Rule: One Action = One Job

Every time you call an Action:
1. Spark looks at the DAG for this RDD
2. Optimizes the plan
3. Splits into Stages
4. Creates Tasks
5. Runs them on executors
6. Returns a result to the Driver

This means calling `rdd.count()` **twice** runs the job **twice**!
Solution: **cache** the RDD if you'll use it multiple times.

---

## Complete List of RDD Actions

| Action | Returns | Description |
|--------|---------|-------------|
| `collect()` | Python list | ALL elements → Driver (⚠️ OOM risk) |
| `count()` | Long | Number of elements |
| `first()` | Single element | First element of RDD |
| `take(n)` | Python list | First n elements |
| `takeSample(f, n, seed)` | Python list | n random elements |
| `takeOrdered(n, key=f)` | Python list | Smallest n elements (or by key) |
| `top(n, key=f)` | Python list | Largest n elements |
| `reduce(f)` | Single value | Combine all elements with f |
| `fold(zero, f)` | Single value | reduce with zero/identity value |
| `aggregate(zero, seqOp, combOp)` | Any type | Most powerful aggregation |
| `sum()` | Numeric | Sum all elements |
| `max()` | Numeric/Comparable | Max element |
| `min()` | Numeric/Comparable | Min element |
| `mean()` | Float | Average |
| `stdev()` | Float | Standard deviation |
| `variance()` | Float | Variance |
| `stats()` | StatCounter | count, mean, stdev, max, min |
| `countByValue()` | Dict | Frequency count per unique value |
| `foreach(f)` | None | Apply f to each element (side effect) |
| `foreachPartition(f)` | None | Apply f to each partition (side effect) |
| `saveAsTextFile(path)` | None | Write RDD as text files |
| `saveAsPickleFile(path)` | None | Write RDD as Python pickle |

---

## `collect()` — The Most Dangerous Action

```
⚠️  WARNING ⚠️

collect() brings ALL data from ALL executors to the Driver's RAM.

With 10GB of RDD data:
  → Driver needs 10GB free RAM
  → If Driver has only 2GB → OutOfMemoryError → job fails
  → In production, this kills the Driver process

Safe alternatives:
  rdd.take(100)                    → first 100 elements
  rdd.first()                      → first 1 element  
  rdd.takeSample(False, 1000)      → random 1000 elements
  rdd.count()                      → just the count (scalar)
```

**When is `collect()` safe?**
- After filtering to a known-small result
- After aggregation (e.g., `groupByKey` result with 10 keys)
- In tests on small data
- After `take()` or `limit()`

---

## `reduce()` vs `fold()` vs `aggregate()`

These three often confuse beginners. Here's the difference:

```
reduce(f):
  → f takes 2 elements, returns 1 element of the SAME type
  → No zero value → fails on empty RDD
  → Example: reduce(lambda a, b: a + b)  → sum

fold(zero, f):
  → Same as reduce but with a zero/identity value
  → Safe on empty RDD (returns zero)
  → zero must be identity for f: for +, zero=0; for *, zero=1; for max, zero=float('-inf')
  → Example: fold(0, lambda a, b: a + b)  → sum with safety

aggregate(zero, seqOp, combOp):
  → Most powerful: can change TYPE!
  → seqOp: (accumulator, element) → new_accumulator  [within partition]
  → combOp: (acc1, acc2) → merged_accumulator  [between partitions]
  → Example: compute (sum, count) simultaneously → (int, int) from int RDD
```

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week2 - RDD Actions") \
    .master("local[4]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("SparkSession ready!")

## Part 1: `collect()`, `count()`, `first()`, `take()`

In [ ]:
rdd = sc.parallelize([15, 3, 8, 22, 1, 47, 6, 33, 9, 14, 55, 2], 3)

print("=== Basic Actions ===")
print(f"collect()         : {rdd.collect()}")
print(f"count()           : {rdd.count()}")
print(f"first()           : {rdd.first()}")
print(f"take(4)           : {rdd.take(4)}")
print(f"takeOrdered(4)    : {rdd.takeOrdered(4)}")
print(f"top(4)            : {rdd.top(4)}")
print()
print("Note: takeOrdered returns SMALLEST first (ascending)")
print("      top() returns LARGEST first (descending)")

In [ ]:
# takeOrdered and top with a custom key function
words = sc.parallelize(["spark", "rdd", "hadoop", "map", "filter", "action", "job"])

print("takeOrdered by word length (shortest first):")
print(words.takeOrdered(3, key=lambda w: len(w)))

print("top by word length (longest first):")
print(words.top(3, key=lambda w: len(w)))

print("takeOrdered alphabetically:")
print(words.takeOrdered(3))

In [ ]:
# takeSample — random sample (non-deterministic without seed)
import time

rdd = sc.parallelize(range(1, 101), 4)

# Without seed — different result each run
sample1 = rdd.takeSample(withReplacement=False, num=10)
print(f"Random sample (no seed):  {sorted(sample1)}")

# With seed — reproducible (great for testing)
sample2 = rdd.takeSample(withReplacement=False, num=10, seed=42)
sample3 = rdd.takeSample(withReplacement=False, num=10, seed=42)
print(f"Seeded sample (run 1):    {sorted(sample2)}")
print(f"Seeded sample (run 2):    {sorted(sample3)}")
print(f"Same result: {sample2 == sample3}")

## Part 2: Numeric Actions — `sum()`, `max()`, `min()`, `mean()`, `stats()`

In [ ]:
salaries = sc.parallelize([95000, 88000, 72000, 68000, 55000, 62000, 105000, 78000], 2)

print("=== Numeric Actions ===")
print(f"count()   : {salaries.count()}")
print(f"sum()     : {salaries.sum():,.0f}")
print(f"max()     : {salaries.max():,.0f}")
print(f"min()     : {salaries.min():,.0f}")
print(f"mean()    : {salaries.mean():,.2f}")
print(f"stdev()   : {salaries.stdev():,.2f}")
print(f"variance(): {salaries.variance():,.2f}")
print()
# stats() gives all of the above in one pass (one job!)
stats = salaries.stats()
print("stats() — all in one pass:")
print(f"  {stats}")
print(f"  count   = {stats.count()}")
print(f"  mean    = {stats.mean():,.2f}")
print(f"  stdev   = {stats.stdev():,.2f}")

## Part 3: `reduce()`, `fold()`, `aggregate()`

In [ ]:
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 4)

# reduce: combine all elements pairwise into one result
# Works like: ((((1+2)+3)+4)+5)... but distributed across partitions
total = numbers.reduce(lambda a, b: a + b)
product = numbers.reduce(lambda a, b: a * b)
max_val = numbers.reduce(lambda a, b: max(a, b))

print("=== reduce() ===")
print(f"sum via reduce:     {total}")
print(f"product via reduce: {product}")
print(f"max via reduce:     {max_val}")

# How reduce actually works (distributed):
# Partition 0: [1,2,3]    → reduce → 6
# Partition 1: [4,5,6]    → reduce → 15
# Partition 2: [7,8,9]    → reduce → 24
# Partition 3: [10]       → reduce → 10
# Final: 6 + 15 + 24 + 10 = 55
print(f"\nManual verify: {sum(range(1, 11))}")

In [ ]:
# fold: reduce with a zero/identity element
# Safe on empty RDDs!
empty_rdd = sc.parallelize([], 2)

try:
    # reduce on empty RDD raises ValueError
    empty_rdd.reduce(lambda a, b: a + b)
except Exception as e:
    print(f"reduce() on empty RDD: {type(e).__name__}: {e}")

# fold is safe on empty RDD (returns zero)
result = empty_rdd.fold(0, lambda a, b: a + b)
print(f"fold(0) on empty RDD: {result}  ← safe!")

# fold on normal data
numbers = sc.parallelize([1, 2, 3, 4, 5], 2)
print(f"fold(0, sum): {numbers.fold(0, lambda a, b: a + b)}")  # 15
print(f"fold(1, mul): {numbers.fold(1, lambda a, b: a * b)}")  # 120

In [ ]:
# aggregate: most powerful — can change TYPE!
# seqOp: combine element into accumulator (within partition)
# combOp: combine two accumulators (between partitions)

# Example: compute (sum, count) simultaneously in one pass
# This is like computing mean without calling mean() and sum() separately (2 jobs)
# aggregate does it in ONE job!

numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 3)

zero_value = (0, 0)  # (sum, count)

def seq_op(acc, element):
    """Combine element into accumulator within a partition."""
    return (acc[0] + element, acc[1] + 1)

def comb_op(acc1, acc2):
    """Combine two partition accumulators."""
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

total_sum, total_count = numbers.aggregate(zero_value, seq_op, comb_op)
mean = total_sum / total_count

print("=== aggregate() — compute sum AND count in one pass ===")
print(f"sum={total_sum}, count={total_count}, mean={mean}")
print(f"Verify with stats: mean={numbers.mean()}, count={numbers.count()}, sum={numbers.sum()}")

In [ ]:
# aggregate: powerful example — find (min, max, sum) in one pass
numbers = sc.parallelize([5, 3, 8, 1, 9, 2, 7, 4, 6], 3)

import math
zero = (math.inf, -math.inf, 0)  # (min, max, sum)

def seq_op2(acc, x):
    return (min(acc[0], x), max(acc[1], x), acc[2] + x)

def comb_op2(a1, a2):
    return (min(a1[0], a2[0]), max(a1[1], a2[1]), a1[2] + a2[2])

min_val, max_val, total = numbers.aggregate(zero, seq_op2, comb_op2)
print(f"In one pass: min={min_val}, max={max_val}, sum={total}")
print(f"Verify: min={numbers.min()}, max={numbers.max()}, sum={numbers.sum()}")

## Part 4: `countByValue()` — Frequency Count

In [ ]:
# countByValue — returns a Python dict of {value: count}
# ⚠️ This returns to Driver! Only use when cardinality is LOW (few unique values)

log_levels = sc.parallelize([
    "ERROR", "INFO", "ERROR", "WARN", "INFO", "ERROR",
    "INFO", "DEBUG", "ERROR", "WARN", "INFO", "ERROR",
], 3)

freq = log_levels.countByValue()
print("Log level frequencies (countByValue):")
for level, count in sorted(freq.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {level:<6} {count:3d}  {bar}")

# When to use: low cardinality (few unique values)
# When NOT to use: high cardinality (millions of unique values — use reduceByKey instead)

## Part 5: `foreach()` and `foreachPartition()` — Side Effects

These actions execute a function on each element or partition **on the executor**.
They return nothing — used purely for side effects (writing to DBs, external services).

```
Key rule: Variables you create inside foreach run on EXECUTORS.
You CANNOT return data to the Driver with foreach.
You CANNOT modify Driver-side variables inside foreach (use Accumulators for that).
```

In [ ]:
# foreach: runs function on each element ON THE EXECUTOR
# Classic mistake: trying to collect results via append to a Driver-side list

rdd = sc.parallelize([1, 2, 3, 4, 5], 2)

# This looks like it would work but WON'T (in a real cluster):
driver_list = []  # This list lives on the Driver

# On a real cluster, this runs on EXECUTORS — they can't modify driver_list!
# In local mode it appears to work (same process) but don't rely on this.
rdd.foreach(lambda x: driver_list.append(x))
print(f"driver_list (unreliable in real cluster): {driver_list}")
print("In a real cluster this would be EMPTY — executors can't modify Driver variables!")

# Correct use: writing to external system
print("\nCorrect foreach usage (simulated DB write):")
def write_to_db(record):
    # In production: db_connection.execute(INSERT INTO ..., record)
    print(f"  [Executor] Writing record: {record}")

rdd.foreach(write_to_db)

In [ ]:
# foreachPartition: ONE function call per partition (vs one per element)
# Use for: DB writes (open connection once per partition, write many rows)

rdd = sc.parallelize(range(1, 13), 4)

def write_partition_to_db(partition_iterator):
    records = list(partition_iterator)
    # In production:
    # conn = psycopg2.connect(...)   # connect ONCE
    # cursor.executemany(sql, records)   # write ALL records
    # conn.close()
    print(f"  [Executor] Partition batch: {records} ({len(records)} records)")

print("foreachPartition — one batch per partition:")
rdd.foreachPartition(write_partition_to_db)

## Part 6: Action Cost — One Action = One Full Job

In [ ]:
import time

rdd = sc.parallelize(range(1, 1_000_001), 4)  # 1 million numbers

print("=== Action Cost Demo ===")
print("Each action below triggers a FULL job (read + process + return)")
print()

t0 = time.time()
count = rdd.count()
print(f"count()  = {count}  ({time.time()-t0:.3f}s) ← Job 1")

t0 = time.time()
total = rdd.sum()
print(f"sum()    = {total}  ({time.time()-t0:.3f}s) ← Job 2 (re-processes same data!)")

t0 = time.time()
mean = rdd.mean()
print(f"mean()   = {mean}  ({time.time()-t0:.3f}s) ← Job 3")

print()
print("= 3 separate jobs, each reading and processing all 1M elements")
print()
print("Solution: CACHE the RDD if you'll call multiple actions on it")
print("  rdd.cache()  # mark for caching")
print("  rdd.count()  # Job 1: processes + caches in memory")
print("  rdd.sum()    # Job 2: reads from CACHE — much faster!")

In [ ]:
# Cache comparison
rdd = sc.parallelize(range(1, 1_000_001), 4)
rdd_cached = rdd.cache()

print("Without cache (2 jobs, each reprocesses data):")
t0 = time.time()
rdd.count(); rdd.sum()
print(f"  Time: {time.time()-t0:.3f}s")

print("\nWith cache (1 process job + 1 fast job):")
t0 = time.time()
rdd_cached.count()   # first action: process + cache
rdd_cached.sum()     # second action: read from cache
print(f"  Time: {time.time()-t0:.3f}s")

rdd_cached.unpersist()  # release memory

## Part 7: `saveAsTextFile()` — Writing RDD to Disk

In [ ]:
import os, shutil

result_rdd = sc.parallelize(["line one", "line two", "line three", "line four"], 2)

out_path = "/tmp/rdd_output"
if os.path.exists(out_path):
    shutil.rmtree(out_path)

result_rdd.saveAsTextFile(out_path)  # ← This is an ACTION

print("Files written:")
for f in sorted(os.listdir(out_path)):
    print(f"  {out_path}/{f}")

# Read it back to verify
print("\nContent of each part file:")
for f in sorted(os.listdir(out_path)):
    if f.startswith("part-"):
        with open(f"{out_path}/{f}") as file:
            content = file.read().strip()
            print(f"  {f}: {repr(content)}")

print()
print("Notice: one part file per partition!")
print("This is why you see 'part-00000', 'part-00001' etc in S3/HDFS outputs.")
print("Number of output files = number of partitions")

## Interview Cheat Sheet

**Q: What is an Action in Spark?**
> An Action is an operation that triggers the execution of the DAG (all accumulated lazy transformations) and returns a result — either to the Driver (collect, count, first) or to storage (saveAsTextFile). Each Action = one Spark Job.

**Q: Why is `collect()` dangerous?**
> `collect()` brings ALL data from all executors back to the Driver's RAM. On a large RDD (millions of rows, GBs of data), this causes an OutOfMemoryError on the Driver. Use `take(n)`, `first()`, or `count()` instead for inspecting data.

**Q: What is the difference between `reduce()` and `fold()`?**
> Both combine elements pairwise. `reduce()` fails on an empty RDD because there's no initial value. `fold()` takes a zero/identity value, making it safe on empty RDDs. For sum: `reduce(+)` or `fold(0, +)`. For product: `reduce(*)` or `fold(1, *)`.

**Q: When would you use `aggregate()` instead of `reduce()`?**
> When you need to compute a different TYPE than the RDD's element type, or compute multiple aggregates in one pass. Example: RDD of integers, but you want (sum, count) — a tuple. `aggregate(zero=(0,0), seqOp=accumulate, combOp=merge)` does this in one job instead of calling `sum()` and `count()` separately (two jobs).

**Q: What's the difference between `foreach()` and `foreachPartition()`?**
> `foreach(f)` calls `f` once per element on the executor. `foreachPartition(f)` calls `f` once per partition with an iterator over all elements. Use `foreachPartition` when setup cost is high (DB connection, file handle) — you amortize that cost across all elements in the partition instead of paying it per element.

**Q: If I call `count()` twice on the same RDD, how many jobs are triggered?**
> Two jobs — each Action re-executes the entire DAG. Unless the RDD is cached (`rdd.cache()`), in which case the second `count()` reads from memory.

## Exercises

1. Create an RDD of 1000 random integers (0-999). Find min, max, mean, and count using `stats()`. Then do the same using `aggregate()` in one pass.
2. Use `takeSample(False, 20, seed=7)` to get a reproducible sample of 20 elements from `range(1, 1001)`. Verify it's always the same with the same seed.
3. Create an RDD of words. Use `countByValue()` to count frequency. Then get top 5 most frequent with `takeOrdered(5, key=lambda x: -x[1])` after converting to pair RDD.
4. Time the difference between calling `.sum()` and `.count()` separately vs using `aggregate()` for both in one pass.

In [ ]:
# Exercise 1: stats() vs aggregate()
import random
random.seed(42)
rnd = sc.parallelize([random.randint(0, 999) for _ in range(1000)], 4)

stats = rnd.stats()
print("Exercise 1 via stats():")
print(f"  count={stats.count()}, min={stats.min()}, max={stats.max()}, mean={stats.mean():.2f}")

import math
zero = (math.inf, -math.inf, 0, 0)  # (min, max, sum, count)

def seq(acc, x): return (min(acc[0],x), max(acc[1],x), acc[2]+x, acc[3]+1)
def comb(a, b): return (min(a[0],b[0]), max(a[1],b[1]), a[2]+b[2], a[3]+b[3])

mn, mx, s, cnt = rnd.aggregate(zero, seq, comb)
print(f"\nVia aggregate(): count={cnt}, min={mn}, max={mx}, mean={s/cnt:.2f}")